# Read the counters of each function by layer

You'll build, for each C, C++ or Python function of each app on each system, the distribution of its duration and of each counter over all its calls, and put the functions side by side in one table.
It takes about 10 minutes of your time on a compute node.

## Before you start

Run the kernel inside an allocation and open the full sweep, as in [Measure the flop rate as processes share a node](06-flop-rate-as-processes-share-a-node.ipynb).

In [1]:
from dftracer.utils import DataFrame, Indexer, TraceViewer, col

TRACE = "/usr/workspace/genesis-wisdom/dftracer-traces/genesis-sweep/genesis.pfw.gz"
INDEX = "../data/full-index"
with Indexer(files=[TRACE], index_dir=INDEX) as ix:
    ix.ensure_indexed()
    runs = ix.rowset("runs")
tv = TraceViewer(TRACE, INDEX)
print(runs.shape)

(9624, 10)


## 1. Give each function a layer

Each `func` record has the `cat` of its calls.
The categories of C and C++ functions, I/O, MPI and the GPU runtimes are known, and a Python annotation sets a category of its own, so every other category is Python.

In [2]:
KNOWN = {
    "C_APP": "C",
    "CPP_APP": "C++",
    "STDIO": "I/O",
    "POSIX": "I/O",
    "p2p": "MPI",
    "comm": "MPI",
    "collective": "MPI",
    "KERNEL_DISPATCH": "GPU",
    "MEMORY_COPY": "GPU",
    "PAGE_MIGRATION": "GPU",
    "SCRATCH_MEMORY": "GPU",
    "env": "other",
}


def layer_of(cat):
    if cat in KNOWN:
        return KNOWN[cat]
    if cat.startswith(("CUDA_", "HIP_")):
        return "GPU"
    return "Python"


func = tv.filter(col("gtype") == "func").collect()
cats = func.groupby("cat").size()["cat"].to_list()
layers = DataFrame.from_dict({"cat": cats, "layer": [layer_of(c) for c in cats]})
func = func.merge(layers, on="cat")
display(func.groupby("layer").size())

,layer,size
0,C,432291
1,C++,98430
2,GPU,398041
3,I/O,86339
4,MPI,1272155
5,other,4806


The table keeps the application functions, the C, C++ and Python layers.
This sweep has no Python layer, because its applications are C and C++.

## 2. Pick the counters

A counter means the same thing only on one CPU, so each row of the table is one function of one app on one system.
Each run measured one set of hardware counters, and the table takes each counter from the runs that measured it.

In [3]:
COUNTERS = ["PAPI_TOT_INS", "PAPI_TOT_CYC", "PAPI_FP_OPS", "PAPI_L1_DCM", "PAPI_BR_MSP", "cpu.user_pct"]
APP = ["C", "C++", "Python"]

fns = func[func["layer"].isin(APP)].merge(runs[["run", "system", "app"]], on="run")
print(fns.shape)

(530721, 22)


These are the call paths of the C and C++ functions in all runs, each with the system and the app of its run.

## 3. Read the counters of the functions

A `counter` record holds the value credited to each call of one call path for one counter.
Read the counters in one scan, since the counter records are most of the file, and keep the ones of the application functions.

In [4]:
ctr = tv.filter((col("gtype") == "counter") & col("metric").isin(COUNTERS)).collect()
ctr = ctr.merge(fns[["run", "path", "system", "app", "layer", "cat"]], on=["run", "path"])
print(ctr.shape)

(955316, 22)


Each row is one counter of one call path in one run.

## 4. Fold the distribution of each counter

Fold the records of each function and counter: add the calls, take the least and the greatest value, weight the mean of each record by its calls and merge the sketches for the percentiles.
Fold the durations of the same functions the same way, so one table holds the duration and every counter.

In [5]:
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.float_format", "{:.4g}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

KEY = ["system", "app", "layer", "cat", "name"]
Q = {"p25": 0.25, "p50": 0.5, "p75": 0.75, "p90": 0.9, "p99": 0.99}
STATS = ["min", "p25", "p50", "mean", "p75", "p90", "p99", "max"]

counters = ctr.groupby([*KEY, "metric"]).agg(
    calls=col("v.n").sum(),
    total=(col("v.avg") * col("v.n")).sum(),
    min=col("v.min").min(),
    **{name: col("v.sketch").sketch_quantile(q) for name, q in Q.items()},
    max=col("v.max").max(),
)
durations = fns.groupby(KEY).agg(
    calls=col("count").sum(),
    total=col("dur.sum").sum(),
    min=col("dur.min").min(),
    **{name: col("dur.sketch").sketch_quantile(q) for name, q in Q.items()},
    max=col("dur.max").max(),
)
dist = pd.concat([durations.to_pandas().assign(metric="dur"), counters.to_pandas()])
dist["mean"] = dist["total"] / dist["calls"]
one = dist[(dist["system"] == "tuolumne") & (dist["name"] == "hypre_CSRMatrixMatvecOutOfPlace")].set_index("metric")
display(one[["calls", *STATS]])

,calls,min,p25,p50,mean,p75,p90,p99,max
metric,,,,,,,,,
dur,45438467,100,308,685.6,1.145e+04,4317,2.509e+04,3.517e+05,1.906e+06
PAPI_BR_MSP,5037419,0.8799,1686,4584,5.241e+04,2.014e+04,7.39e+04,7.371e+05,7.685e+06
PAPI_FP_OPS,5049982,4.857,3.057e+05,8.146e+05,2.05e+07,5.557e+06,4.36e+07,8.415e+08,9.124e+08
PAPI_L1_DCM,5048621,10.66,5.698e+04,1.43e+05,3.641e+06,9.56e+05,7.653e+06,1.507e+08,1.78e+08
PAPI_TOT_CYC,5036862,7.742e+04,1.122e+06,2.547e+06,4.203e+07,1.572e+07,9.324e+07,1.307e+09,1.679e+09
PAPI_TOT_INS,5036862,1.001e+05,2.351e+06,4.831e+06,7.572e+07,2.923e+07,1.699e+08,2.579e+09,2.945e+09
cpu.user_pct,45426048,0.1653,5.755,8.936,10.19,14.16,18.73,23.81,46.73


This is the distribution of one function on tuolumne, the matrix-vector product of amg: one row for its duration and one for each counter.
`dur` is in microseconds, the hardware counters count events in one call, and `cpu.user_pct` is the user time of the node in percent while the call ran.
`calls` is smaller for the hardware counters, because each run measured one set of them, so a counter comes only from the runs of its set.
The calls spread over four orders of magnitude, from a median of 686 microseconds to 0.35 s at the 99th percentile, and the counts per call spread with them.
The mean call takes 11 ms, 17 times the median, so a few long calls hold most of the time.

## 5. Put the functions side by side

`pivot_table` turns the long table into one row for each function and a group of columns for each counter, with every statistic in the group.
Show the duration, the instructions and the user time here.

In [6]:
wide = dist.pivot_table(index=KEY, columns="metric", values=STATS)
wide = wide.swaplevel(axis=1).reindex(columns=pd.MultiIndex.from_product([["dur", *COUNTERS], STATS]))
order = durations.to_pandas().sort_values(["system", "app", "layer", "total"], ascending=[True, True, True, False])
top = order.groupby(["system", "app", "layer"]).head(10).set_index(KEY).index
display(wide.loc[top, ["dur", "PAPI_TOT_INS", "cpu.user_pct"]])

dur                                                                       PAPI_TOT_INS                                                                       cpu.user_pct  \
                                                                       min       p25       p50      mean       p75       p90       p99       max          min       p25       p50      mean       p75       p90       p99       max          min   
system   app    layer cat     name                                                                                                                                                                                                                 
matrix   amg    C     C_APP   main                                    3636 4.382e+05 1.078e+06 1.689e+07 3.651e+06 2.698e+07 2.075e+08 1.614e+09    5.447e+05 3.482e+09 1.025e+10 1.081e+11 3.916e+10 9.826e+10 1.297e+12 9.927e+12       0.2244   
                              hypre_GMRESSolve                         645      5712 1.751e+04 9.136e+05 2.311e+05 1.426e+06 2.039e+07 1.407e+08    1.697e+04  6.25e+07 1.878e+08 6.013e+09 1.444e+09 8.564e+09 1.353e+11 9.915e+11       0.2218   
                              hypre_ParCSRMatrixMatvecOutOfPlace       100     206.5     788.6 1.447e+04      7558 2.887e+04 4.941e+05 8.206e+05            0 1.545e+06 5.339e+06 8.733e+07 3.716e+07 1.185e+08 3.482e+09 7.027e+09       0.2216   
                              hypre_BoomerAMGSolve                     113     837.4      1755 8.808e+04 3.387e+04 1.678e+05  3.05e+06 4.424e+06    1.962e+04 9.347e+06 2.165e+07 5.621e+08 2.075e+08 1.255e+09 2.065e+10 3.217e+10       0.2216   
                              hypre_BoomerAMGCycle                     106     820.8      1755 8.807e+04 3.387e+04 1.678e+05  3.05e+06 4.424e+06            0 9.162e+06 2.165e+07  5.62e+08 2.075e+08 1.255e+09 2.065e+10 3.217e+10       0.2216   
                              hypre_CSRMatrixMatvecOutOfPlace          100     354.3      1003 1.302e+04      6977 2.665e+04 4.382e+05 7.722e+05            0 2.399e+06 5.447e+06 7.581e+07 3.362e+07 1.072e+08 3.279e+09 3.785e+09       0.2216   
                              hypre_ParCSRRelax                        100     179.5       728 1.495e+04      5712 3.065e+04 5.353e+05 9.256e+05         5915 1.317e+06 4.735e+06 8.869e+07 3.642e+07  1.03e+08 3.697e+09 7.317e+09       0.2216   
                              hypre_GMRESSetup                        1038 4.573e+04 7.691e+04 1.178e+06 6.157e+05 2.086e+06 4.106e+07 4.632e+07    2.097e+05 4.618e+08 8.085e+08 7.405e+09 4.338e+09 1.759e+10  1.72e+11 3.238e+11       0.2215   
                              hypre_PCGSolve                          1597 3.255e+04 1.347e+05 1.987e+06 1.514e+06 6.141e+06  2.08e+07 2.098e+07     1.27e+06 4.618e+08  1.36e+09 1.094e+10 7.749e+09 4.595e+10 5.613e+10 3.325e+11       0.2587   
                              hypre_BoomerAMGCoarsenPMIS               100      2144      3985 4.097e+04      6977  2.46e+04 7.371e+05 6.949e+06    2.708e+04  1.92e+07 3.791e+07 2.711e+08 6.907e+07 1.994e+08 5.299e+09 3.046e+10       0.2215   
         laghos C++   CPP_APP main                               2.943e+06 4.819e+07 1.284e+08 1.605e+08 2.293e+08 3.561e+08  5.99e+08 7.439e+08    1.622e+10 4.065e+11 1.105e+12 1.391e+12 1.974e+12 3.065e+12 5.366e+12 7.649e+12        1.079   
                              Mult                                    2203      8025      9048 1.056e+04 1.083e+04 1.751e+04 2.411e+04 2.734e+06    1.616e+06 6.907e+07 7.945e+07 9.309e+07   9.9e+07 1.537e+08 2.248e+08  2.32e+10        1.079   
                              SolveVelocity                           2139      7558      8694 1.011e+04 1.041e+04 1.682e+04 2.363e+04 2.734e+06    1.117e+06 6.636e+07 7.634e+07  8.91e+07 9.512e+07 1.507e+08  2.16e+08 2.319e+10        1.079   
                              SolveEnergy                              100     354.3     391.6       436     441.5     518.1      1065 2.058e+06    1.189e+05 2.989e+06

Each app on each system shows its ten functions with the most time.
tuolumne has no laghos rows, since laghos didn't run there.
matrix has no value for `PAPI_L1_DCM`, since no counter set of matrix holds it.
The same function changes with the system: the median call of `Mult` in laghos takes 9 ms on matrix and 28 ms on tioga.
`wide` holds every counter, so pick others by name in the column list, such as `["PAPI_L1_DCM"]`.

## 6. Read the counters as rates

Counts per call grow with the length of the call, so divide them by the cycles to compare functions of different sizes.
Use the sums over all calls, so each rate counts every call once.

In [7]:
total = ctr.groupby([*KEY, "metric"]).agg(total=col("v.sum").sum())
total = total.pivot_table(values="total", index=KEY, columns="metric").to_pandas()
total = total.set_index(KEY)
rates = pd.DataFrame(
    {
        "ipc": total["PAPI_TOT_INS"] / total["PAPI_TOT_CYC"],
        "flops_per_cycle": total["PAPI_FP_OPS"] / total["PAPI_TOT_CYC"],
        "l1_miss_per_kinst": 1000 * total["PAPI_L1_DCM"] / total["PAPI_TOT_INS"],
        "br_miss_per_kinst": 1000 * total["PAPI_BR_MSP"] / total["PAPI_TOT_INS"],
    }
)
display(rates.loc[top].round(3))

ipc  flops_per_cycle  l1_miss_per_kinst  br_miss_per_kinst
system   app    layer cat     name                                                                                           
matrix   amg    C     C_APP   main                               1.669            0.263                NaN              1.754
                              hypre_GMRESSolve                   1.703            0.343                NaN              0.745
                              hypre_ParCSRMatrixMatvecOutOfPlace 1.614            0.343                NaN              0.643
                              hypre_BoomerAMGSolve               1.639             0.33                NaN              0.915
                              hypre_BoomerAMGCycle               1.639             0.33                NaN              0.915
                              hypre_CSRMatrixMatvecOutOfPlace    1.556            0.351                NaN               0.64
                              hypre_ParCSRRelax                  1.587            0.342                NaN              0.478
                              hypre_GMRESSetup                   1.634            0.026                NaN              6.157
                              hypre_PCGSolve                     1.373            0.186                NaN              0.586
                              hypre_BoomerAMGCoarsenPMIS         1.672            0.006                NaN              4.686
         laghos C++   CPP_APP main                                2.47            0.001                NaN              0.239
                              Mult                               2.484            0.001                NaN              0.238
                              SolveVelocity                      2.484            0.001                NaN              0.238
                              SolveEnergy                        2.493            0.001                NaN               0.23
                              GetTimeStepEstimate                2.669                0                NaN              0.078
                              LagrangianHydroOperator            0.426                0                NaN              0.244
                              UpdateQuadratureData               1.162            0.001                NaN              0.288
                              Rho0DetJ0Vol                       0.375                0                NaN              0.294
                              InternalEnergy                     0.685                0                NaN                0.2
                              SetEssentialTrueDofs               0.681                0                NaN              0.271
         minife C++   CPP_APP driver                             0.454                0                NaN               0.07
                              cg_solve                           0.574                0                NaN              0.077
                              make_local_matrix                  0.337                0                NaN              0.056
                              dot                                0.582                0                NaN              0.077
                              generate_matrix_structure          0.469                0                NaN              0.078
                              operator()                         0.606                0                NaN              0.072
                              finish_exchange_externals          0.641                0                NaN              0.066
                              assemble_FE_data                    0.36                0                NaN              0.067
                              perform_element_loop_cuda           0.36                0                NaN              0.067
                              waxpby                             0.533                0                NaN              0.088
tioga    amg    C     C_APP   main 

The solve functions of amg in C, such as `hypre_GMRESSolve` and `hypre_ParCSRRelax`, run at 1.6 to 2 instructions per cycle on all three systems.
Their flop rate is 0.33 to 0.49 flops per cycle on matrix and tuolumne, and 0.1 to 0.18 on tioga.
The setup functions of amg, such as `hypre_GMRESSetup`, do almost no flops and miss 4 to 6 branches per thousand instructions on matrix and tuolumne.
laghos in C++ runs its solve at about 2.5 instructions per cycle on matrix and 2 on tioga, with almost no flops on the CPU.
minife in C++ runs at 0.3 to 0.6 instructions per cycle on matrix, about 0.9 on tuolumne and 0.8 to 2.6 on tioga.
On tuolumne minife offloads its solve to the GPU, so its CPU counters count the host side, which mostly waits.
Rows such as `hypre_BoomerAMGSolve` and `hypre_BoomerAMGCycle` match because one only calls the other.

## Limits

A counter is read per process and shared out among the calls by their time, so the rate of a short function is mostly the rate of the code around it.
Read the rates of long functions, such as a solve, as their own, and the rates of short ones as the rate of their phase.
A rate that needs two counters comes from the runs whose counter set holds both, so a system whose sets split them has no value there.
The records keep no sum of squares, so the table has no variance; the spread shows in the percentiles.

## What you built

You built one table of the counters of each application function, by system, app and layer, and turned the counts into rates.

- The same sketch merge in duql, `quantile(merge(v.sketch), q)`: [duql reference](https://dftracer.readthedocs.io/projects/utils/en/latest/reference/duql.html)
- Back to the start: [Count the runs in a genesis trace](01-count-the-runs.ipynb)